In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Visualisasi timing circuit

<Accordion>
<AccordionItem title="Versi paket">

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Meskipun [timeline drawer](/guides/visualize-circuit-timing) yang sudah ada di Qiskit berguna untuk circuit statis, ini mungkin tidak mencerminkan timing [dynamic circuit](/guides/classical-feedforward-and-control-flow) secara akurat karena operasi implisit seperti broadcasting dan penentuan cabang. Sebagai bagian dari dukungan dynamic circuit, Qiskit Runtime mengembalikan informasi timing circuit yang akurat di dalam hasil job saat diminta.

> **Note:** - Ini adalah fungsi eksperimental. Statusnya preview release dan karenanya bisa berubah.
> - Fungsi ini hanya berlaku untuk job Qiskit Runtime Sampler.
> - Meskipun total waktu circuit dikembalikan dalam metadata "compilation", ini BUKAN waktu yang digunakan untuk penagihan (waktu QPU).

### Aktifkan pengambilan data timing
Untuk mengaktifkan pengambilan data timing, atur flag eksperimental `scheduler_timing` ke `True` saat menjalankan job primitif.

In [1]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Akses data timing circuit
Saat diminta, data timing circuit untuk setiap PUB dikembalikan dalam metadata hasil job, di bawah `["compilation"]["scheduler_timing"]["timing"]`. Field ini berisi informasi timing mentah. Untuk menampilkan informasi timing, gunakan alat visualisasi bawaan, seperti yang dijelaskan di bagian [Visualisasikan timing-nya](#visualize-timings).

Gunakan kode berikut untuk mengakses data timing circuit untuk PUB pertama:

In [2]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Memahami data timing mentah
Meskipun memvisualisasikan data timing circuit dengan metode `draw_circuit_schedule_timing` adalah kasus penggunaan yang paling umum, mungkin berguna untuk memahami struktur data timing mentah yang dikembalikan. Ini bisa membantumu, misalnya, untuk mengekstrak informasi secara terprogram.

Data timing yang dikembalikan dalam `["compilation"]["scheduler_timing"]["timing"]` adalah daftar string. Setiap string merepresentasikan satu instruksi pada suatu channel dan dipisahkan koma menjadi tipe data berikut:

- `Branch` - Menentukan apakah instruksi berada dalam alur kontrol (then / else) atau cabang utama.
- `Instruction` - Gate dan qubit yang dioperasikan.
- `Channel` - Channel yang ditetapkan dengan instruksi. Bisa salah satu dari berikut:
      - `Qubit x` - Channel drive untuk qubit _x_.
      - `AWGRx_y` (arbitrary waveform generator readout) - Digunakan oleh channel readout untuk berkomunikasi saat mengukur qubit. Argumen _x_ dan _y_ bersesuaian dengan ID instrumen readout dan nomor qubit.
- `T0` - Waktu mulai instruksi dalam jadwal lengkap.
- `Duration` - Durasi instruksi, dalam satuan _dt_ detik, di mana 1 dt = 1 siklus penjadwalan. Kamu bisa menemukan nilai `dt` sebuah backend menggunakan [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - Jenis operasi pulse yang digunakan.

Contoh:

In [3]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Visualisasikan timing-nya
Dengan `qiskit-ibm-runtime` v0.43.0 atau lebih baru, kamu bisa memvisualisasikan timing circuit. Untuk memvisualisasikan timing, kamu perlu mengubah metadata hasil ke `fig` menggunakan [metode `draw_circuit_schedule_timing`](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26). Metode ini mengembalikan gambar `plotly`, yang bisa kamu tampilkan langsung, simpan ke file, atau keduanya. Untuk informasi lebih lanjut tentang perintah `plotly` yang digunakan, lihat [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) dan [`fig.write_image("<path.format>")`](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html).

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d8287kugbeec73alfbug (DONE)


![Mengarahkan kursor ke output menampilkan informasi seperti waktu mulai, selesai, dan durasi.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Contoh gambar yang dihasilkan')

#### Memahami gambar yang dihasilkan
Gambar output data timing circuit dari `draw_circuit_schedule_timing` menyampaikan informasi berikut:

- Sumbu X adalah waktu dalam satuan _dt_ detik, di mana 1 dt = 1 siklus penjadwalan. Kamu bisa menemukan nilai `dt` sebuah backend menggunakan [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- Sumbu Y adalah channel (bayangkan channel sebagai instrumen yang memancarkan pulse).
    - `Receive channel` - Satu-satunya channel yang bukan instrumen sendiri. Ini adalah instruksi yang dimainkan di semua channel yang merupakan bagian dari prosedur komunikasi dengan hub pada saat itu.
    - `Qubit x` - Channel drive untuk qubit x.
    - `AWGRx_y` (arbitrary waveform generator readout) - Digunakan oleh channel readout untuk berkomunikasi saat mengukur qubit. Argumen _x_ dan _y_ bersesuaian dengan ID instrumen readout dan nomor qubit.
    - `Hub` - Mengontrol broadcasting.

Selain itu, setiap instruksi memiliki format *X_Y*, di mana *X* adalah nama instruksi dan *Y* adalah jenis pulse. Tipe `play` menerapkan pulse kontrol, dan `capture` merekam keadaan qubit. Kamu juga bisa mengarahkan kursor ke setiap instruksi untuk mendapatkan detail lebih lanjut. Misalnya, gambar sebelumnya menunjukkan pulse kontrol untuk gate X yang diterapkan pada qubit 10 pada 1161 dt.

### Contoh end-to-end
Contoh ini menunjukkan cara mengaktifkan opsi, mendapatkan informasi timing circuit dari metadata, dan menampilkannya sebagai gambar.

Pertama, siapkan lingkungan, definisikan circuit dan ubah ke circuit ISA, lalu definisikan dan jalankan job.

In [5]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,1365,0,shift_phase\nmain,sx_0,Qubit 0,1365,9,play\nmain,sx_0,Qubit 0,1369,0,shift_phase\nmain,rz_0,Qubit 0,1374,0,shift_phase\nmain,barrier,Qubit 0,1374,0,barrier\nmain,measure_0,Qubit 0,1374,64,play\nmain,measure_0,Qubit 0,1438,108,play\nmain,measure_0,AWGR0_0,1485,360,capture\nmain,measure_0,Qubit 0,1546,64,play\nmain,measure_0,Qubit 0,1610,64,play\nmain,barrier,Qubit 0,2046,0,barrier\nmain,broadcast,Hub,1485,561,broadcast\nmain,receive,Receive,2046,7,receive\nthen,x_0,Qubit 0,2061,9,play\nmain,barrier,Qubit 0,2079,0,barrier\nmain,measure_0,Qubit 0,2079,64,play\nmain,measure_0,Qubit 0,2143,108,play\nmain,measure_0,AWGR0_0,2190,360,capture\nmain,measure_0,Qubit 0,2251,64,play\nmain,measure_0,Qubit 0,2315,64,play\nmain,barrier,Qubit 0,2725,0,barrier\nmain,barrier,Qubit 0,2725,0,barrier\n'

Finally, you can visualize and save the timing:

In [6]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Selanjutnya, dapatkan timing jadwal circuit: